In [2]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import optuna
from sklearn.model_selection import cross_val_score

# For showing all the the Data Frame columns
%matplotlib inline
pd.set_option('display.max_columns', None) 

In [3]:
from utils import *

In [4]:
df_s = pd.read_csv('../data/reduced_data.csv')

In [5]:
df_s = df_s.drop(['timestamp', 'LPS', 'Pressure_switch', 'Oil_level', 
                  'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'], axis=1).reset_index(drop=True)

In [6]:
df_s.isna().sum()

TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
DV_eletric         0
Towers             0
failure            0
month              0
day                0
dtype: int64

In [7]:
# Select target and features
X = df_s.drop('failure', axis=1)
y = df_s['failure'].astype(int)

In [8]:
y.dtype

dtype('int64')

In [9]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df_s) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [10]:
print(y_test.unique())

[0 1]


In [11]:
# I chose sampling_strategy = 0.05 because is the biggest Recall
smote = SMOTE(sampling_strategy=0.05, random_state=42) 
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [23]:
# Fit the model
rf_sm = RandomForestClassifier(
    n_estimators=189,
    max_depth=10,    
    min_samples_split=4,     
    min_samples_leaf=30, 
    max_features=None,  
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_sm.fit(X_train_sm, y_train_sm)

,n_estimators,189
,criterion,'gini'
,max_depth,10
,min_samples_split,4
,min_samples_leaf,30
,min_weight_fraction_leaf,0.0
,max_features,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [31]:
y_pred = rf_sm.predict(X_test)
print("Rando Forest Report:\n", classification_report(y_test, y_pred, zero_division=0))

Rando Forest Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     75546
           1       0.35      0.31      0.33       270

    accuracy                           1.00     75816
   macro avg       0.67      0.65      0.66     75816
weighted avg       1.00      1.00      1.00     75816



In [30]:
y_prob = rf_sm.predict_proba(X_test)[:, 1]
# Default threshold is 0.5 — too strict for rare failures. 
# Prioritizes Recall (catching real failures) over Precision (avoiding false alarms)
# At 0.15, we flag failure if model is ≥15% confident
y_pred = (y_prob >= 0.15).astype(int) 
print("Rando Forest Report:\n", classification_report(y_test, y_pred, zero_division=0))

Rando Forest Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     75546
           1       0.24      0.99      0.39       270

    accuracy                           0.99     75816
   macro avg       0.62      0.99      0.69     75816
weighted avg       1.00      0.99      0.99     75816

